# Stage 0 — avatar emotion feasibility test (RunPod)

Runs the **Stage 0 feasibility test** for adding emotional facial expression to the
Speech2Face avatar. It does **not** touch the streaming pipeline: it loads the shipped
renderer checkpoints read-only and writes only into `stage0_out/`.

## What this answers

| # | question | output |
|---|---|---|
| 1 | Do the three expression videos give emotion **directions** in the 32-D motion latent that are *expression*, not *framing*? | `r_null`, direction cosines |
| 2 | How far can each direction be pushed before the **narrow** (distilled) renderer stops tracking the reference? | `alpha_max` per emotion |
| 3 | Does adding a direction to a talking trajectory destroy **lip-sync**? | mouth-motion retention |

Final output is a **GO / GO-STAGE-3-FIRST / NO-GO** verdict.

## Why it is cheap

Stage 0 needs **no PersonaPlex trunk** — no 19 GB download, **no gated Hugging Face
licence, no HF token**. It needs only two public files (`ref.jpeg`,
`renderer_narrow.pt`) from `niloy629/personaplex-parallel-depformer`, about 250 MB.

## Encoder modes (important)

Mapping video frames to 32-D latents needs an image→latent encoder.

* **`full`** — `latent_token_encoder` from the full IMTalker renderer
  (`weights/renderer.ckpt`). Gives an *independent* reference for the sweep.
  **`prepare_x86.sh` does not download this file** — production never loads it
  (`TRY19_NO_FULL_RENDERER=1`), so it is not in the public asset list. Supply it
  yourself if you have it.
* **`invert`** — optimise codes *through the narrow renderer* (public weights only).
  For question 2 this is arguably the **more direct** measurement: it searches the
  renderer's whole 32-D input space, so the residual it reaches is an upper bound on
  how well that renderer can represent the expression. What it cannot provide is an
  independent reference for direction purity.

`auto` (the default) uses `full` if the checkpoint loads, else `invert`. The notebook
reports which mode ran, and the verdict is annotated with it.

## Requirements

* any Ampere-or-newer NVIDIA GPU (this test is small: ~2 GB VRAM)
* the repo's `emotion_demo_videos/` (happy, sad, angry, neutral) and
  `tools/stage0_emotion_dirs.py`, both of which must be committed to the repo

## 0 · Settings

In [ ]:
import os, sys, json, time, shutil, subprocess, urllib.request, hashlib, textwrap
from pathlib import Path

# ----------------------------------------------------------------- settings
REPO_URL = "https://github.com/MoshiHead/speech2face_orin_n_original_emotion_v1.git"
REPO_DIR = "/workspace/speech2face_emotion"   # the clone lands here
BRANCH   = ""                                  # "" = the repo's default branch

# Environment strategy:
#   "auto"    -> use this pod's Python if it already has torch+CUDA, pip-installing
#                only the few missing bits (fast, ~1 min). Falls back to "full".
#   "minimal" -> force the fast path.
#   "full"    -> build ./env from scratch with setup_env_x86.sh (~8-20 min). Use this
#                if the pod's torch is broken or you also want to run the server.
ENV_MODE = "auto"

# Encoder mode for the test: "auto" | "full" | "invert"   (see the intro)
ENCODER = "auto"

# Stage 0 knobs
OUT_DIR   = "stage0_out"
ALPHAS    = "0,0.25,0.5,0.75,1.0,1.25,1.5,2.0"
CORR_THRESH = 0.80      # narrow-vs-reference residual correlation defining alpha_max
DYN_ALPHA = 1.0         # alpha used for the lip-sync retention test
NARROW_DTYPE = "fp32"   # production runs fp16; fp32 isolates renderer range from fp16 noise

# An HF token is NOT needed for Stage 0 (all required weights are public). It is only
# read if present, in case you later want the gated trunk in this same pod.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN and os.path.exists("/workspace/hf_token.txt"):
    HF_TOKEN = open("/workspace/hf_token.txt").read().strip()

OURS_REPO = "niloy629/personaplex-parallel-depformer"   # public
NEEDED_PUBLIC = ["ref.jpeg", "renderer_narrow.pt"]      # all Stage 0 needs
EMOTION_VIDEOS = ["neutral.mp4", "happy.mp4", "sad.mp4", "angry.mp4"]


def sh(cmd, cwd=None, check=True, env=None, quiet=False):
    """Run a shell command, streaming its output into the notebook as it happens."""
    cwd = cwd or (REPO_DIR if os.path.isdir(REPO_DIR) else os.getcwd())
    if not quiet:
        print("$ " + cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         errors="replace", env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError("command failed (exit %d): %s" % (rc, cmd))
    return rc


def head(title):
    print("\n" + "=" * 72 + "\n  " + title + "\n" + "=" * 72, flush=True)


print("repo       :", REPO_URL)
print("into       :", REPO_DIR)
print("env mode   :", ENV_MODE)
print("encoder    :", ENCODER)
print("HF token   :", ("present (%d chars, not needed for Stage 0)" % len(HF_TOKEN))
      if HF_TOKEN else "absent (fine - Stage 0 needs no gated weights)")

## 1 · GPU and disk check

In [ ]:
head("GPU / disk")
sh("nvidia-smi --query-gpu=index,name,compute_cap,memory.total,driver_version "
   "--format=csv || echo 'NO GPU VISIBLE'", cwd="/", check=False)
sh("df -h /workspace / 2>/dev/null | sed -n '1,5p'; uname -m", cwd="/", check=False)

out = subprocess.run(["bash", "-c", "nvidia-smi --query-gpu=compute_cap,memory.total "
                      "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
if out.strip():
    cap, mem = [x.strip() for x in out.strip().splitlines()[0].split(",")]
    gib = float(mem) / 1024
    print("\ncompute capability %s, %.0f GiB VRAM" % (cap, gib))
    if float(cap) < 8.0:
        print("!! Ampere or newer (>= 8.0) recommended; Stage 0 may still run")
    elif gib < 8:
        print("!! under 8 GiB - set NARROW_DTYPE='fp16' if you hit CUDA OOM")
    else:
        print("ok - Stage 0 needs only ~2 GB")
else:
    print("!! no GPU visible - Stage 0 needs CUDA (you can pass --device cpu but it will be very slow)")

## 2 · System packages

`ffmpeg` and `libgl1` are what OpenCV needs to decode the `.mp4` videos. Skipped
automatically if already present.

In [ ]:
head("system packages")
need_apt = subprocess.run(["bash", "-c", "command -v ffmpeg >/dev/null && "
                           "ldconfig -p | grep -q libGL.so.1"], capture_output=True).returncode != 0
if need_apt:
    sh("export DEBIAN_FRONTEND=noninteractive && apt-get update -qq && "
       "apt-get install -y -qq --no-install-recommends "
       "  git curl ca-certificates ffmpeg libsndfile1 libgl1 libglib2.0-0 "
       "&& echo '--- installed ---' && ffmpeg -version | head -1", cwd="/", check=False)
else:
    print("ffmpeg + libGL already present - skipping apt")

## 3 · Clone the repository

In [ ]:
head("clone / update repo")
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(REPO_DIR, "already exists - updating")
    # FETCH_HEAD, not origin/HEAD: a --depth 1 clone has no remote HEAD ref
    sh("git -C %s fetch --depth 1 origin %s && git -C %s reset --hard FETCH_HEAD"
       % (REPO_DIR, BRANCH or "HEAD", REPO_DIR), cwd="/", check=False)
else:
    branch = ("--branch " + BRANCH) if BRANCH else ""
    sh("git clone --depth 1 %s %s %s" % (branch, REPO_URL, REPO_DIR), cwd="/")

os.chdir(REPO_DIR)
print("\ncwd:", os.getcwd())
sh("ls -la", check=False)

# the two Stage 0 inputs that must be in the repo
print()
for p in ["tools/stage0_emotion_dirs.py"] + ["emotion_demo_videos/" + v for v in EMOTION_VIDEOS]:
    print(("  OK      " if os.path.exists(p) else "  MISSING ") + p)

If anything above says **MISSING**, the file is not committed to the repo. Either commit
it and re-run the previous cell, or upload it with the Jupyter file browser into
`/workspace/speech2face_emotion/` (the videos are only ~2 MB in total).

## 4 · Python environment

Stage 0 needs `torch` (CUDA), `numpy`, `opencv`, `Pillow` — plus `torchvision`, `timm`
and `matplotlib` only for `full` encoder mode. RunPod's PyTorch images already ship
torch, so the fast path just pip-installs the few missing pieces.

In [ ]:
head("python environment")
MIN_IMPORTS = "import torch, numpy, cv2, PIL; assert torch.cuda.is_available()"
FULL_IMPORTS = MIN_IMPORTS + "; import torchvision, timm, matplotlib"


def probe(py, expr):
    """True if `py` can run `expr`."""
    if not py:
        return False
    return subprocess.run([py, "-c", expr], capture_output=True).returncode == 0


def find_python():
    cands = []
    for p in (os.path.join(REPO_DIR, "env/bin/python"), ".python_path"):
        if p == ".python_path" and os.path.exists(p):
            cands.append(open(p).read().strip())
        elif os.path.exists(p):
            cands.append(p)
    cands += [sys.executable, shutil.which("python3") or "", shutil.which("python") or ""]
    seen, uniq = set(), []
    for c in cands:
        if c and c not in seen:
            seen.add(c); uniq.append(c)
    return uniq


PY = None
mode = ENV_MODE
cands = find_python()
print("candidate interpreters:")
for c in cands:
    print("   ", c, "->", "torch+CUDA OK" if probe(c, MIN_IMPORTS) else "not usable as-is")

if mode in ("auto", "minimal"):
    # pick the first interpreter that has torch+CUDA (with or without cv2)
    base = next((c for c in cands if probe(c, "import torch; assert torch.cuda.is_available()")), None)
    if base is None:
        print("\nno interpreter with torch+CUDA found", end=" ")
        if mode == "minimal":
            raise RuntimeError("ENV_MODE='minimal' but no torch+CUDA interpreter exists; use ENV_MODE='full'")
        print("-> falling back to ENV_MODE='full'")
        mode = "full"
    else:
        print("\nusing:", base)
        missing = [m for m, imp in (("opencv-python-headless", "import cv2"),
                                    ("Pillow", "import PIL"),
                                    ("timm", "import timm"),
                                    ("matplotlib", "import matplotlib"),
                                    ("huggingface_hub", "import huggingface_hub"))
                   if not probe(base, imp)]
        if missing:
            print("installing:", " ".join(missing))
            sh('"%s" -m pip install -q %s' % (base, " ".join(missing)), check=False)
        else:
            print("nothing to install")
        PY = base
        mode = "minimal"

if mode == "full":
    if not os.path.exists(os.path.join(REPO_DIR, "env/bin/python")):
        t0 = time.time()
        sh("chmod +x setup_env_x86.sh && ./setup_env_x86.sh")
        print("\n--- setup_env_x86.sh took %.1f min ---" % ((time.time() - t0) / 60))
    PY = os.path.join(REPO_DIR, "env/bin/python")
    sh('"%s" -m pip install -q -r requirements-x86-extra.txt' % PY, check=False)

Path(".stage0_python").write_text(PY)
print("\nPY =", PY)

In [ ]:
head("verify imports")
# Written to a file rather than a heredoc: the body contains %-format specifiers
# of its own, so building it with %-formatting would misfire.
Path(".stage0_verify.py").write_text(textwrap.dedent("""
    import sys, torch, numpy, cv2, PIL
    print("python     ", sys.version.split()[0])
    print("torch      ", torch.__version__, "CUDA", torch.version.cuda,
          "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")
    print("numpy      ", numpy.__version__)
    print("opencv     ", cv2.__version__)
    print("Pillow     ", PIL.__version__)
    for m in ("torchvision", "timm", "matplotlib"):
        try:
            __import__(m)
            print("%-11s OK (full encoder mode available)" % m)
        except Exception as e:
            print("%-11s MISSING -> invert encoder mode only (%s)" % (m, type(e).__name__))
    if not torch.cuda.is_available():
        print("\\n!! torch cannot see a CUDA GPU - Stage 0 will not run on this interpreter")
"""))
sh('"%s" .stage0_verify.py' % PY)

## 5 · Download the weights (public, ~250 MB, no token)

Mirrors `prepare_x86.sh`'s `fetch()`: same public repo, same `SHA256SUMS`
verification. Only the two files Stage 0 actually needs are downloaded, and the
download is resumable — `huggingface_hub` caches, so re-running skips finished files.

In [ ]:
head("public avatar weights")
os.makedirs("weights", exist_ok=True)

FETCH = textwrap.dedent("""
    import os, sys, hashlib
    from huggingface_hub import hf_hub_download
    REPO, FOLDER, DEST = sys.argv[1], sys.argv[2], sys.argv[3]
    files = sys.argv[4:]
    os.makedirs(DEST, exist_ok=True)
    want = {}
    for line in open(hf_hub_download(REPO, FOLDER + "/SHA256SUMS")):
        h, f = line.split()
        want[os.path.basename(f)] = h


    def sha256(path):
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for c in iter(lambda: fh.read(1 << 24), b""):
                h.update(c)
        return h.hexdigest()


    for f in files:
        dst = os.path.join(DEST, f)
        real = os.path.realpath(dst)
        if os.path.exists(real) and sha256(real) == want[f]:
            print("    " + f + ": already present, sha256 OK")
            continue
        src = hf_hub_download(REPO, FOLDER + "/" + f)
        if sha256(src) != want[f]:
            raise SystemExit("hash mismatch for " + f + " - clear the HF cache and retry")
        if os.path.islink(dst) or os.path.exists(dst):
            os.remove(dst)
        os.symlink(src, dst)
        print("    " + f + ": downloaded, sha256 OK -> " + dst)
""")
Path(".fetch_assets.py").write_text(FETCH)

env = {"HF_HOME": os.path.join(REPO_DIR, "hf")}
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN
# RunPod images set HF_HUB_ENABLE_HF_TRANSFER=1; huggingface_hub then refuses to
# download unless hf_transfer is importable. Turn it off rather than fail.
if os.environ.get("HF_HUB_ENABLE_HF_TRANSFER", "0") != "0":
    if subprocess.run([PY, "-c", "import hf_transfer"], capture_output=True).returncode != 0:
        print("    note: HF_HUB_ENABLE_HF_TRANSFER=1 but hf_transfer missing -> disabling")
        env["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

ok = False
for attempt in (1, 2, 3):
    rc = sh('"%s" .fetch_assets.py %s avatar_assets weights %s'
            % (PY, OURS_REPO, " ".join(NEEDED_PUBLIC)), check=False, env=env)
    if rc == 0:
        ok = True
        break
    print("\n!! attempt %d failed; retrying in 10s" % attempt)
    time.sleep(10)
if not ok:
    raise RuntimeError("could not download the public avatar weights - check network/DNS to huggingface.co")

sh("ls -lL weights/", check=False)

### Optional: the full IMTalker renderer (`weights/renderer.ckpt`)

Not published in `avatar_assets`, because production never loads it. If you have it,
put it at `weights/renderer.ckpt` (upload, or copy from another pod/volume) **before**
running the test — the notebook will then use `full` encoder mode and give an
independent reference for the sweep. Without it the test runs in `invert` mode, which
still answers all three questions.

In [ ]:
head("full renderer (optional)")
p = "weights/renderer.ckpt"
if os.path.exists(p):
    print("FOUND %s (%.1f MB) -> 'full' encoder mode is available"
          % (p, os.path.getsize(os.path.realpath(p)) / 1e6))
else:
    print("not present:", p)
    print("  -> the test will run in 'invert' encoder mode (public weights only).")
    print("  -> this is expected on a fresh pod; prepare_x86.sh does not download it.")
    # a couple of places it might already exist on this machine
    sh("ls -l assets/renderer.ckpt /workspace/renderer.ckpt 2>/dev/null || true", check=False)

## 6 · Preflight — every input the test needs

In [ ]:
head("preflight")
REQUIRED = {
    "test script":      ["tools/stage0_emotion_dirs.py"],
    "emotion videos":   ["emotion_demo_videos/" + v for v in EMOTION_VIDEOS],
    "reference image":  ["weights/ref.jpeg"],
    "narrow renderer":  ["weights/renderer_narrow.pt"],
}
OPTIONAL = {"full renderer (optional)": ["weights/renderer.ckpt"]}

missing = []
for group, paths in list(REQUIRED.items()) + list(OPTIONAL.items()):
    print("\n%s:" % group)
    for p in paths:
        exists = os.path.exists(p)
        size = (os.path.getsize(os.path.realpath(p)) / 1e6) if exists else 0
        print("   %s %-42s %s" % ("OK     " if exists else "MISSING",
                                  p, ("%.1f MB" % size) if exists else ""))
        if not exists and group in REQUIRED:
            missing.append(p)

# the videos must actually decode, and all four must agree on frame geometry
if not missing:
    Path(".stage0_probe_videos.py").write_text(textwrap.dedent("""
        import cv2, sys
        print()
        bad = 0
        for v in sys.argv[1:]:
            p = "emotion_demo_videos/" + v
            c = cv2.VideoCapture(p)
            n = int(c.get(cv2.CAP_PROP_FRAME_COUNT)); fps = c.get(cv2.CAP_PROP_FPS)
            w = int(c.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(c.get(cv2.CAP_PROP_FRAME_HEIGHT))
            ok, _ = c.read(); c.release()
            print("   %-12s %4d frames  %.2f fps  %dx%d  decode=%s"
                  % (v, n, fps, w, h, "OK" if ok else "FAILED"))
            if not ok:
                bad += 1
        if bad:
            print("\\n!! %d video(s) failed to decode - is ffmpeg/libGL installed?" % bad)
    """))
    sh('"%s" .stage0_probe_videos.py %s' % (PY, " ".join(EMOTION_VIDEOS)), check=False)

print()
if missing:
    print("!! CANNOT RUN - missing %d required file(s):" % len(missing))
    for p in missing:
        print("     ", p)
    print("\n   videos/script -> commit them to the repo or upload into", REPO_DIR)
    print("   weights       -> re-run the download cell above")
else:
    print("PREFLIGHT PASSED - ready to run Stage 0")

## 7 · Run the Stage 0 test

This does all of it in one pass:

1. **L1 framing correction** — per-video similarity warp onto `ref.jpeg`'s framing,
   estimated from each video's *median* frame (so expression and mouth motion are
   marginalised out), constrained to translation + rotation + uniform scale.
2. **encode** all frames to 32-D motion latents (`full` or `invert` mode).
3. **L2 nuisance projection** — PCA of latent deltas from *known* similarity and
   lighting perturbations of `ref.jpeg` (zero expression content by construction),
   projected out of every direction.
4. **L3 speech projection** — PCA of the neutral video's own temporal variance (constant
   expression ⇒ its variance is mouth + pose), projected out to protect lip-sync.
5. **directions** `d_e = mean(video e) − mean(neutral)` for happy / sad / angry.
6. **checks** `r_null` and the direction cosines.
7. **alpha sweep** across intensities, rendered through the narrow renderer and the
   reference, giving `alpha_max` per emotion.
8. **lip-sync retention** on a talking trajectory with the offset applied.

If it hits CUDA OOM the next cell retries in fp16 automatically.

In [ ]:
head("Stage 0 run")
PY = Path(".stage0_python").read_text().strip()

ARGS = ("--videos_dir emotion_demo_videos "
        "--ref_path weights/ref.jpeg "
        "--renderer_path weights/renderer.ckpt "
        "--distilled_renderer weights/renderer_narrow.pt "
        "--out_dir %s --alphas %s --corr_thresh %s --dyn_alpha %s "
        "--narrow_dtype %s --encoder %s --device cuda"
        % (OUT_DIR, ALPHAS, CORR_THRESH, DYN_ALPHA, NARROW_DTYPE, ENCODER))

t0 = time.time()
rc = sh('"%s" -u tools/stage0_emotion_dirs.py %s' % (PY, ARGS), check=False)

if rc != 0:
    print("\n!! first attempt failed (exit %d) - retrying in fp16 with smaller settings" % rc)
    ARGS_FP16 = ARGS.replace("--narrow_dtype %s" % NARROW_DTYPE, "--narrow_dtype fp16")
    rc = sh('"%s" -u tools/stage0_emotion_dirs.py %s --invert_frames 16' % (PY, ARGS_FP16),
            check=False)

print("\n--- Stage 0 took %.1f min (exit %d) ---" % ((time.time() - t0) / 60, rc))
if rc != 0:
    print("!! the test did not complete. Scroll up for the traceback; nothing below will be valid.")

## 8 · Results — expression sweeps

In each sheet the **top row is the reference** (what the direction *should* look like)
and the **bottom row is the narrow renderer** at increasing intensity. Read it by asking
where the bottom row stops following the top row, and where it starts to smear or clip —
that is `alpha_max`.

In [ ]:
from IPython.display import Image, display, Markdown
head("expression sweeps")
for e in ("happy", "sad", "angry"):
    p = os.path.join(OUT_DIR, "sweep_%s.png" % e)
    if os.path.exists(p):
        display(Markdown("### %s" % e))
        display(Image(filename=p))
    else:
        print("missing:", p)

## 9 · Results — framing correction (L1)

These are the per-video median frames **after** alignment onto `ref.jpeg`. They should
all show the same head size and position. If one is visibly offset or scaled
differently, L1 failed for that video and its direction is suspect.

In [ ]:
head("aligned medians")
imgs = [("reference", "weights/ref.jpeg")] + [
    (n, os.path.join(OUT_DIR, "aligned_median_%s.png" % n))
    for n in ("neutral", "happy", "sad", "angry")]
for name, p in imgs:
    if os.path.exists(p):
        display(Markdown("**%s** — `%s`" % (name, p)))
        display(Image(filename=p, width=320))
    else:
        print("missing:", p)

## 10 · Summary and verdict

In [ ]:
head("Stage 0 summary")
rp = os.path.join(OUT_DIR, "stage0_report.json")
if not os.path.exists(rp):
    print("!! %s not found - the test did not finish. Nothing to summarise." % rp)
else:
    R = json.load(open(rp))
    mode = R.get("encoder_mode", "?")

    print("encoder mode : %s" % mode)
    if mode != "full":
        print("               (reference = the narrow renderer's own best fit,")
        print("                not an independent model - see the intro)")

    print("\n--- videos / framing correction (L1) " + "-" * 34)
    print("  %-9s %6s %8s %9s %9s  %s" % ("video", "frames", "scale", "tx", "ty", "ECC"))
    for n, v in R["videos"].items():
        al = v.get("align", {})
        print("  %-9s %6d %8.4f %9.2f %9.2f  %s" % (
            n, v["frames"], al.get("scale", float("nan")),
            al.get("tx", 0.0), al.get("ty", 0.0), str(al.get("ecc"))[:22]))

    print("\n--- projections " + "-" * 55)
    nu, sp = R.get("nuisance", {}), R.get("speech", {})
    print("  nuisance subspace : k=%s  explained=%s" % (
        nu.get("k"), [round(x, 3) for x in nu.get("explained", [])[:4]]))
    print("  speech subspace   : k=%s  source=%s" % (sp.get("k"), sp.get("source")))
    print("                      explained=%s" % [round(x, 3) for x in sp.get("explained", [])[:4]])

    print("\n--- directions " + "-" * 56)
    print("  %-7s %10s %10s %10s   %s" % ("emotion", "|raw|", "|after N|", "|final|", "removed"))
    for e, d in R["directions"].items():
        print("  %-7s %10.4f %10.4f %10.4f   N %4.1f%%  S %4.1f%%" % (
            e, d["norm_raw"], d["norm_after_nuisance"], d["norm_final"],
            100 * d["frac_removed_nuisance"], 100 * d["frac_removed_speech"]))

    print("\n--- CHECK 1 : r_null (framing really removed?) " + "-" * 24)
    rn = R["check_r_null"]
    print("  r_null = %.3f   %s" % (rn["r_null"], rn["verdict"]))
    print("  mean(neutral latents) and encode(ref.jpeg) are BOTH neutral expression,")
    print("  so a working pipeline collapses their difference.  < 0.35 good, > 0.6 bad.")

    print("\n--- CHECK 2 : direction cosines " + "-" * 39)
    for k, v in R["check_cosines"]["cos"].items():
        print("  cos(%-14s) = %+.3f" % (k, v))
    print("  -> %s" % R["check_cosines"]["verdict"])
    print("  expected: happy|sad < 0 (opposite valence), sad|angry > 0 (shared negative)")

    print("\n--- intensity sweep : alpha_max " + "-" * 39)
    for e, s in R["sweep"].items():
        print("  %-7s alpha_max = %-5g" % (e, s["alpha_max"]))
        print("     %6s %7s %10s %8s %8s" % ("alpha", "corr", "E_narrow", "gain", "clip%"))
        for row in s["per_alpha"]:
            print("     %6g %7.3f %10.2f %8.3f %8.2f" % (
                row["alpha"], row["residual_corr"], row["narrow_energy"],
                row["gain_ratio"] if row["gain_ratio"] == row["gain_ratio"] else 0.0,
                100 * row["narrow_clip_frac"]))

    print("\n--- lip-sync retention (alpha=%s) " % R["lipsync"]["alpha"] + "-" * 30)
    for e, v in R["lipsync"]["retention"].items():
        print("  %-7s %.3f   %s" % (e, v, "OK" if v >= 0.9 else "DEGRADED"))
    print("  mask = top 5%% most temporally variable pixels (%.2f%% of frame)"
          % (100 * R["lipsync"]["mask_frac"]))

    print("\n" + "=" * 72)
    print("  VERDICT : %s" % R.get("verdict"))
    print("=" * 72)

### How to read the verdict

| verdict | meaning | next step |
|---|---|---|
| **GO → Stage 1** | `r_null < 0.35`, cosines not all-parallel, `alpha_max ≥ 1.0` for happy+sad, lip-sync ≥ 0.9 | latent steering works as-is — implement the runtime offset |
| **GO, Stage 3 first** | directions transfer but `alpha_max < 0.5` | the narrow renderer clips; it needs the emotion head (the blink-style conditioning) before it can be expressive |
| **NO-GO** | `r_null` large, or all cosines > 0.9 | extraction failed — framing dominates; fall back to landmark-based alignment (adds `mediapipe`) |
| **PARTIAL** | mixed | read the per-emotion `alpha_max` and lip-sync rows above |

Read `r_null` **first**: if framing was not removed, nothing else in the report is
trustworthy.

## 11 · Optional extra runs

Useful but not required for the verdict.

* **fp16** — production runs the narrow renderer in fp16 on both Orin and RunPod, so
  this checks that `alpha_max` survives the precision actually used in the pipeline.
* **ablation** — turns L2 and L3 off, showing how much the projections actually bought.

In [ ]:
head("fp16 variant (production precision)")
rc = sh('"%s" -u tools/stage0_emotion_dirs.py %s'
        % (PY, ARGS.replace("--narrow_dtype %s" % NARROW_DTYPE, "--narrow_dtype fp16")
                   .replace("--out_dir %s" % OUT_DIR, "--out_dir %s_fp16" % OUT_DIR)),
        check=False)
print("\nexit", rc)

In [ ]:
head("ablation: no nuisance / no speech projection")
rc = sh('"%s" -u tools/stage0_emotion_dirs.py %s --no_nuisance --no_speech'
        % (PY, ARGS.replace("--out_dir %s" % OUT_DIR, "--out_dir %s_ablation" % OUT_DIR)),
        check=False)
print("\nexit", rc)

# r_null with and without the projections is the headline comparison
for tag, d in (("with L2+L3", OUT_DIR), ("ablation", "%s_ablation" % OUT_DIR)):
    p = os.path.join(d, "stage0_report.json")
    if os.path.exists(p):
        R = json.load(open(p))
        print("  %-11s r_null=%.3f  alpha_max=%s" % (
            tag, R["check_r_null"]["r_null"],
            {e: s["alpha_max"] for e, s in R["sweep"].items()}))

## 12 · Package the results for download

In [ ]:
head("package results")
stamp = time.strftime("%Y%m%d_%H%M%S")
tarball = "stage0_results_%s.tar.gz" % stamp
dirs = " ".join(d for d in (OUT_DIR, "%s_fp16" % OUT_DIR, "%s_ablation" % OUT_DIR)
                if os.path.isdir(d))
sh("tar czf %s %s" % (tarball, dirs), check=False)
print()
sh("ls -la %s; echo; find %s -type f | sort" % (tarball, dirs), check=False)
print("\nDownload '%s' from the Jupyter file browser (right-click -> Download)." % tarball)
print("It contains: stage0_report.json, emotion_dirs.pt, sweep_*.png, aligned_median_*.png")

## 13 · What to send back

1. the full output of **cell 7** (the Stage 0 run) and **cell 10** (the summary),
2. `stage0_out/stage0_report.json`,
3. the three `stage0_out/sweep_*.png` — specifically, where the narrow row stops
   tracking the reference row.

`stage0_out/emotion_dirs.pt` holds the extracted directions, `ref_x`, both projection
bases and `alpha_max`. That file is the direct input to Stage 1 (the runtime emotion
offset), so keep it.

**Nothing in the streaming pipeline has been modified by this notebook.** It only reads
checkpoints and writes into `stage0_out*/`.